# StateSet Agents v1.0 — Open-Weight RL Suite on Google Colab (AI Ultra H100 / A100 / L4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/open_weight_rl_colab_gpu.ipynb)

> **Built for Google AI Ultra / Colab Pro+ (H100 80GB, A100 40GB/80GB, L4 24GB) with clean T4/CPU fallback.**
>
> What you will run end-to-end in this notebook:
> 1. **Auto-detect GPU tier** (`stateset_agents.utils.colab`) & optionally mount Google Drive for persistent checkpoints.
> 2. **Measure real CUDA Peak VRAM & Triton SRAM Kernel Speedup** (`chunked_gather_token_logprobs` vs. full `[B, T, V]` logit materialization).
> 3. **Run the 6-Domain Verifiable Enterprise Sandbox Pool** (`commerce`, `coding`, `sql`, `rest_api`, `docker_code`, `web_agent`) with per-domain z-score normalization & Turn-Level PRM credit.
> 4. **Train a real Open-Weight HuggingFace Model with LoRA** using **Step-0 PyTorch CAA Warm-Start (`rank1_weight_bake`)** + **Chunked GSPO / FP8 TIS / MoE Aux Loss** + **Online Reward-Hacking Guard**.
> 5. **Export a reproducible SHA-256 GPU evidence bundle** (`v1_ga_evidence.json` + `v1_ga_evidence.md`) + LoRA adapter to Google Drive.

## 1. Install `stateset-agents[training,api]` & Check GPU Runtime

In [ ]:
import os
import subprocess
import sys

# Install from PyPI or pin a release branch/tag (e.g. stateset-agents==1.0.0)
INSTALL_SPEC = os.environ.get(
    "STATESET_INSTALL_SPEC",
    "stateset-agents[training,api] @ git+https://github.com/stateset/stateset-agents.git@release/0.99.1",
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", INSTALL_SPEC])

import stateset_agents
from stateset_agents.utils.colab import detect_colab_runtime

runtime = detect_colab_runtime(probe_cuda=True)
profile = runtime["profile"]
print(f"stateset-agents : {stateset_agents.__version__}")
print(f"GPU             : {runtime['gpu_name']} ({runtime['vram_gb']} GB VRAM)")
print(f"CUDA / PyTorch  : {runtime['cuda_version']} / {runtime['torch_version']}")
print(f"BF16 / Triton   : {runtime['bf16_supported']} / {runtime['triton_available']}")
print(f"Auto-Tuned Tier : {profile['tier']} -> {profile['recommended_model']} (chunk={profile['chunked_loss_chunk_size']}, r={profile['lora_rank']})")

## 2. Optional: Mount Google Drive for Persistent Checkpoints

In [ ]:
from pathlib import Path
from stateset_agents.utils.colab import is_colab_runtime, resolve_checkpoint_dir

MOUNT_GOOGLE_DRIVE = False  # Set True to save LoRA checkpoints & evidence to Drive

if MOUNT_GOOGLE_DRIVE and is_colab_runtime():
    from google.colab import drive
    drive.mount("/content/drive")

RUN_DIR = resolve_checkpoint_dir("stateset-agents/open_weight_rl_v1", fallback_root=Path("outputs"))
print("Artifacts & checkpoints will be saved to:", RUN_DIR)

## 3. Measure Live CUDA Peak VRAM: Full Logits vs. Chunked / Triton LM-Head Kernel

Large-vocabulary open-weight models (`Qwen3`, `V=152,064`, 16k sequences) allocate **37+ GB** just for `[B, T, V]` logits. Here we measure real `torch.cuda.max_memory_allocated()` and exact forward/backward gradient parity on your active Colab GPU.

In [ ]:
import json
from stateset_agents.utils.colab import run_cuda_chunked_vram_benchmark

vram_report = run_cuda_chunked_vram_benchmark()
print(json.dumps(vram_report, indent=2))
assert vram_report["numerical_equivalence"]["exact_match"], "Chunked parity mismatch!"

## 4. Run All 6 Enterprise Verifiable RL Sandboxes Concurrently

In [ ]:
import json
from stateset_agents.utils.colab import run_six_domain_verifiable_eval_sync

sandbox_report = run_six_domain_verifiable_eval_sync()
print(json.dumps(sandbox_report, indent=2))
assert sandbox_report["all_verified"], "Expected 6/6 verifiable sandboxes to pass!"

## 5. Live Open-Weight Model Training: Step-0 PyTorch CAA Warm-Start + Chunked GSPO LoRA

This cell loads a real open-weight Qwen checkpoint onto your Colab Ultra GPU, extracts and bakes a reversible Step-0 CAA steering vector (`rank1_weight_bake`), runs real Chunked GSPO + FP8 TIS optimizer updates, and saves the LoRA adapter to `RUN_DIR`.

> Override `MODEL_ID` to `Qwen/Qwen3-8B` or `Qwen/Qwen3-4B` on Colab Ultra H100/A100 80GB, or keep `Qwen/Qwen2.5-1.5B-Instruct` for a fast 2-minute proof run!

In [ ]:
import json
from stateset_agents.utils.colab import run_live_hf_open_weight_smoke

# Fast default for quick proof; on A100/H100 80GB try profile["recommended_model"]
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" if runtime["cuda_available"] else "Qwen/Qwen2.5-0.5B-Instruct"
STEPS = 6 if runtime["cuda_available"] else 2

hf_report = run_live_hf_open_weight_smoke(
    model_name=MODEL_ID,
    steps=STEPS,
    lora_rank=int(profile["lora_rank"]),
    chunk_size=int(profile["chunked_loss_chunk_size"]),
    output_dir=RUN_DIR,
)
print(json.dumps(hf_report, indent=2))
assert hf_report["improved"], "Expected post-RL margin to beat base margin!"

## 6. Generate Reproducible 3-Seed Ablation & Download Artifacts

In [ ]:
from stateset_agents.utils.colab import (
    capture_v1_evidence,
    is_colab_runtime,
    render_v1_evidence_markdown,
)

evidence = capture_v1_evidence(
    output_dir=RUN_DIR,
    hf_model=None,  # Already captured above; pass MODEL_ID here to bundle into one JSON
    skip_hf_model=True,
    river_mode="offline",
)
print(render_v1_evidence_markdown(evidence))

if is_colab_runtime():
    from google.colab import files
    files.download(str(RUN_DIR / "v1_ga_evidence.md"))